# Etapa 3 — Classificação com SVM

**Inteligência Artificial · Ciência da Computação · PUC Minas · Trabalho Prático 3**

Esta primeira comparação da Etapa 3 avalia uma SVM linear e uma SVM com kernel RBF. A SVM aparece no material de Introdução a Aprendizado de Máquina e no notebook de Bagging, que usa SVC como classificador base.

Abra no Colab e escolha **Ambiente de execução → Executar tudo**. Forneça apenas o **treino.csv** oficial do Canvas, caso seja solicitado. A execução padrão usa CPU e reaproveita o preparo das bibliotecas da Etapa 2. O notebook contém as funções necessárias.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys
import tempfile
from hashlib import sha256
from IPython.display import Markdown, display

candidatos = [
    Path("data/raw/treino.csv"),
    Path("treino.csv"),
    Path("/content/treino.csv"),
]
treino_csv = next((arquivo for arquivo in candidatos if arquivo.is_file()), None)
if treino_csv is None:
    try:
        from google.colab import files
    except ImportError as erro:
        raise FileNotFoundError(
            "Coloque o treino.csv oficial na pasta inicial ou em data/raw e execute novamente."
        ) from erro
    treino_csv = Path("data/raw/treino.csv")
    treino_csv.parent.mkdir(parents=True, exist_ok=True)
    print("Selecione o treino.csv oficial do Canvas.")
    files.upload_file(str(treino_csv))

treino_csv = treino_csv.resolve()
with treino_csv.open(encoding="utf-8-sig", newline="") as arquivo:
    cabecalho = next(csv.reader(arquivo), [])
obrigatorias = {"class", "protocol_type", "service", "flag", "src_bytes", "total_bytes"}
if not obrigatorias.issubset(cabecalho):
    raise ValueError(
        f"O arquivo em {treino_csv} não tem o cabeçalho do treino oficial. "
        "Substitua-o pelo treino.csv correto e execute novamente."
    )
trabalho = Path(tempfile.mkdtemp(prefix="ia-etapa3-svm-"))
codigo_dados = trabalho / "dados_etapa2.py"
codigo_tecnicas = trabalho / "tecnicas_etapa2.py"
codigo_experimentos = trabalho / "executar_svm.py"
print("Base encontrada:", treino_csv.name)

In [ ]:
VERSOES = {
    "numpy": "2.5.4",
    "pandas": "3.0.1",
    "scipy": "1.18.1",
    "scikit-learn": "1.7.2",
    "imbalanced-learn": "0.14.1",
    "joblib": "1.6.0",
    "threadpoolctl": "3.8.0",
    "sklearn-compat": "0.1.6",
}

def consultar_versoes(ambiente):
    consulta = (
        "import importlib.metadata as m, json, sys; "
        "print(json.dumps({n: next((d.version for d in m.distributions(name=n)), None) "
        "for n in sys.argv[1:]}))"
    )
    resposta = subprocess.check_output(
        [sys.executable, "-c", consulta, *VERSOES], env=ambiente, text=True
    )
    return json.loads(resposta)

ambiente = os.environ.copy()
ambiente["PYTHONIOENCODING"] = "utf-8"
ambiente["PYTHONNOUSERSITE"] = "1"
ambiente["OPENBLAS_NUM_THREADS"] = "1"
ambiente["OMP_NUM_THREADS"] = "1"
if consultar_versoes(ambiente) != VERSOES:
    assinatura = sha256(json.dumps(VERSOES, sort_keys=True).encode()).hexdigest()[:12]
    bibliotecas = Path(tempfile.gettempdir()) / f"ia-etapa2-{sys.implementation.cache_tag}-{assinatura}"
    caminho_existente = ambiente.get("PYTHONPATH")
    ambiente["PYTHONPATH"] = os.pathsep.join(filter(None, (str(bibliotecas), caminho_existente)))
    if consultar_versoes(ambiente) != VERSOES:
        print("Instalando as versões dos experimentos. Aguarde a conclusão desta célula.", flush=True)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "--only-binary=:all:", "--upgrade", "--target", str(bibliotecas),
             *[f"{nome}=={versao}" for nome, versao in VERSOES.items()]],
            check=True, env={**os.environ, "PYTHONNOUSERSITE": "1"},
        )
versoes_ativas = consultar_versoes(ambiente)
if versoes_ativas != VERSOES:
    raise RuntimeError(f"As versões do ambiente não coincidiram com as previstas: {versoes_ativas}")
print("Python:", sys.version.split()[0])
print("Bibliotecas:", versoes_ativas)

## Protocolo de comparação

Reutilizamos as cinco divisões estratificadas por grupo da Etapa 2. A busca de configuração usa as quatro primeiras, excluindo a quinta de todos os ajustes e avaliações dessa busca. Na quinta divisão, medimos a melhor opção da busca e, se for diferente, a melhor SVM linear. Essa quinta divisão já foi usada em estudos anteriores do projeto e continua sendo uma verificação interna. O teste oficial fica reservado para a entrega final.

Todas as opções de SVM partem da versão C, com 40 atributos antes da codificação. No treino de cada divisão, removemos duplicatas de mesmo rótulo, imputamos as entradas numéricas pela mediana, marcamos ausências nominais com uma categoria própria, ajustamos o one-hot e aplicamos Z-score às entradas numéricas. A validação preserva os registros e usa somente as transformações aprendidas no treino.

In [ ]:
%%writefile "{codigo_dados}"
"""Prepara os dados e cria divisoes agrupadas para as etapas 2 e 3."""

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

ALVO = "class"
COLUNAS_EXCLUIDAS_EXPERIMENTOS = (
    "num_outbound_cmds", "tempo_resposta_analista_min", "tipo_alerta_soc", "score_ids_legado",
)
SEMENTE = 42
NOMINAIS = ["protocol_type", "service", "flag", "protocolo_cod"]
COPIAS = ["src_bytes_kb", "dst_bytes_kb", "same_srv_pct", "protocolo_cod"]


def carregar_treino(caminho: str | Path) -> pd.DataFrame:
    """Lê o CSV oficial e valida a coluna alvo."""
    caminho = Path(caminho)
    if not caminho.is_file():
        raise FileNotFoundError(f"Base oficial não encontrada: {caminho}")
    dados = pd.read_csv(caminho)
    if ALVO not in dados or dados.empty:
        raise ValueError(f"A base deve ser não vazia e conter {ALVO!r}.")
    if dados[ALVO].isna().any() or not set(dados[ALVO].unique()).issubset({"normal", "anomaly"}):
        raise ValueError("A coluna class deve conter somente normal ou anomaly, sem ausências.")
    return dados.copy()


def identificar_taxas(dados: pd.DataFrame) -> list[str]:
    """Retorna apenas frações ``*_rate``; ``*_pct`` usa escala percentual."""
    return [c for c in dados.select_dtypes(include="number") if c.lower().endswith("rate")]


def preparar_base(dados):
    entradas = dados.drop(columns=["class", *COLUNAS_EXCLUIDAS_EXPERIMENTOS])
    for coluna in NOMINAIS:
        entradas[coluna] = entradas[coluna].astype("string").str.strip().str.casefold().replace("", pd.NA)

    invalidos = {}
    for coluna in ("duration", "src_bytes", "dst_bytes"):
        invalidos[coluna] = entradas[coluna].lt(0)
    for coluna in identificar_taxas(entradas):
        invalidos[coluna] = entradas[coluna].notna() & ~entradas[coluna].between(0, 1)
    invalidos["src_bytes_kb"] = invalidos["src_bytes"] | entradas["src_bytes_kb"].lt(0)
    invalidos["dst_bytes_kb"] = invalidos["dst_bytes"] | entradas["dst_bytes_kb"].lt(0)
    invalidos["total_bytes"] = invalidos["src_bytes"] | invalidos["dst_bytes"] | entradas["total_bytes"].lt(0)
    percentual = entradas["same_srv_pct"]
    invalidos["same_srv_pct"] = invalidos["same_srv_rate"] | (percentual.notna() & ~percentual.between(0, 100))
    for coluna, mascara in invalidos.items():
        entradas[coluna] = entradas[coluna].mask(mascara)

    representacoes = {
        "A": entradas,
        "B": entradas.drop(columns=COPIAS),
        "C": entradas.drop(columns=[*COPIAS, "total_bytes"]),
    }
    assert [m.shape[1] for m in representacoes.values()] == [45, 41, 40]
    grupos = representacoes["C"].groupby(list(representacoes["C"]), dropna=False, sort=False).ngroup()
    assert grupos.notna().all()
    return representacoes, grupos


def escolher_linhas(x, y, indices, regra):
    if regra == "todas":
        return indices
    chaves = x.iloc[indices].assign(rotulo=y[indices])
    mascara = np.ones(len(indices), dtype=bool)
    if regra in {"sem_conflitos", "sem_ambos"}:
        grupos = chaves.groupby(list(x), dropna=False, sort=False)["rotulo"]
        mascara &= grupos.transform("nunique").to_numpy() == 1
    if regra in {"sem_duplicatas", "sem_ambos"}:
        mascara &= ~chaves.duplicated().to_numpy()
    return indices[mascara]


def separar_divisoes(representacoes, y, grupos):
    divisor = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
    particoes = list(divisor.split(representacoes["C"], y, grupos))
    reserva = particoes[-1][1]
    divisoes = []
    for fold, (treino, validacao) in enumerate(particoes, 1):
        if fold != 5:
            treino = np.setdiff1d(treino, reserva)
            assert set(grupos.iloc[treino]).isdisjoint(grupos.iloc[reserva])
            assert set(grupos.iloc[validacao]).isdisjoint(grupos.iloc[reserva])
        assert set(grupos.iloc[treino]).isdisjoint(grupos.iloc[validacao])
        divisoes.append((fold, treino, validacao))
    return divisoes


In [ ]:
%%writefile "{codigo_tecnicas}"
"""Transformacoes ajustadas apenas no treino de cada divisao."""

import numpy as np
import pandas as pd
from sklearn import config_context
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, RobustScaler, StandardScaler
from dados_etapa2 import NOMINAIS, SEMENTE, escolher_linhas


def transformar_numericas(treino, validacao, metodo, escala):
    treino = treino.to_numpy(dtype=np.float64, na_value=np.nan)
    validacao = validacao.to_numpy(dtype=np.float64, na_value=np.nan)
    # Uma entrada sem observacoes no ajuste recebe zero em ambas as parcelas.
    # Valores observados apenas na validacao nao participam do ajuste do KNN.
    if metodo == "knn":
        vazias = np.isnan(treino).all(axis=0)
        treino[:, vazias] = 0.0
        validacao[:, vazias] = 0.0
    if metodo == "mediana_indicador":
        tem_ausencia = np.isnan(treino).any(axis=0)
        indicadores_treino = np.isnan(treino[:, tem_ausencia])
        indicadores_val = np.isnan(validacao[:, tem_ausencia])
    if metodo == "knn":
        # Limita o custo de vizinhança a uma amostra sem rótulos da parte de treino.
        normalizador = StandardScaler().fit(treino)
        amostra = np.random.default_rng(SEMENTE).choice(len(treino), size=min(2000, len(treino)), replace=False)
        imputador = KNNImputer(n_neighbors=5, keep_empty_features=True)
        imputador.fit(normalizador.transform(treino[amostra]))
        with config_context(working_memory=48):
            treino = normalizador.inverse_transform(imputador.transform(normalizador.transform(treino)))
            validacao = normalizador.inverse_transform(imputador.transform(normalizador.transform(validacao)))
    else:
        estrategia = "mean" if metodo == "media" else "median"
        imputador = SimpleImputer(strategy=estrategia, keep_empty_features=True)
        treino, validacao = imputador.fit_transform(treino), imputador.transform(validacao)
    normalizadores = {"zscore": StandardScaler, "robusta": RobustScaler, "minmax": MinMaxScaler}
    if escala in normalizadores:
        normalizador = normalizadores[escala]()
        treino, validacao = normalizador.fit_transform(treino), normalizador.transform(validacao)
    if metodo == "mediana_indicador":
        treino = np.column_stack((treino, indicadores_treino))
        validacao = np.column_stack((validacao, indicadores_val))
    return treino, validacao


def transformar_categoricas(treino, validacao, metodo, codificacao):
    estrategia = "most_frequent" if metodo == "moda" else "constant"
    colunas = list(treino)
    colunas_treino, colunas_validacao = [], []
    for coluna in colunas:
        valores_treino = treino[[coluna]].to_numpy(dtype=object, na_value=np.nan)
        valores_validacao = validacao[[coluna]].to_numpy(dtype=object, na_value=np.nan)
        # A moda nao existe quando toda a coluna do ajuste esta ausente.
        # Nesse caso a categoria explicita mantem a posicao da coluna.
        estrategia_coluna = "constant" if pd.isna(valores_treino).all() else estrategia
        imputador = SimpleImputer(strategy=estrategia_coluna, fill_value="__ausente__")
        colunas_treino.append(imputador.fit_transform(valores_treino))
        colunas_validacao.append(imputador.transform(valores_validacao))
    treino = np.column_stack(colunas_treino)
    validacao = np.column_stack(colunas_validacao)
    if codificacao == "inteiros":
        codificador = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        return codificador.fit_transform(treino), codificador.transform(validacao)
    cod_treino, cod_val = [], []
    for i, coluna in enumerate(colunas):
        agrupar = codificacao == "onehot_raros" and coluna == "service"
        codificador = OneHotEncoder(
            sparse_output=False,
            min_frequency=20 if agrupar else None,
            handle_unknown="infrequent_if_exist" if agrupar else "ignore",
        )
        cod_treino.append(codificador.fit_transform(treino[:, i:i + 1]))
        cod_val.append(codificador.transform(validacao[:, i:i + 1]))
    return np.column_stack(cod_treino), np.column_stack(cod_val)


def transformar(x, y, indices, validacao, config):
    indices = escolher_linhas(x, y, indices, config.linhas)
    treino, validacao = x.iloc[indices], x.iloc[validacao]
    categoricas = [c for c in NOMINAIS if c in x]
    numericas = x.columns.drop(categoricas).tolist()
    num_treino, num_val = transformar_numericas(treino[numericas], validacao[numericas], config.numericos, config.escala)
    cat_treino, cat_val = transformar_categoricas(treino[categoricas], validacao[categoricas], config.nominais, config.codificacao)
    return np.column_stack((num_treino, cat_treino)), np.column_stack((num_val, cat_val)), y[indices]


## Configurações e paralelismo

A versão linear usa LinearSVC e testa C igual a 0,1, 1 e 10. O kernel RBF permite uma fronteira não linear e usa SVC com gamma definido a partir da escala dos dados de treino. Testamos C igual a 1 e 10 no RBF. Cada família tem uma opção adicional com pesos inversamente proporcionais à frequência das classes. A métrica de escolha é F1-macro, com recall de anomaly como desempate.

A célula de execução permite uma ou duas divisões em paralelo com joblib. Cada ajuste usa um thread numérico para reduzir a disputa por recursos. Isso funciona em CPU, sem exigir CUDA. Os tempos são medidos no ambiente onde o notebook roda e podem variar entre execuções.

In [ ]:
%%writefile "{codigo_experimentos}"
"""Compara SVMs nas divisões agrupadas da Etapa 2, usando apenas o treino."""

import json
from pathlib import Path
import sys
from time import perf_counter
from types import SimpleNamespace

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.svm import LinearSVC, SVC
from threadpoolctl import threadpool_limits

from dados_etapa2 import SEMENTE, carregar_treino, preparar_base, separar_divisoes
from tecnicas_etapa2 import transformar

PREPARACAO = SimpleNamespace(
    linhas="sem_duplicatas", numericos="mediana", nominais="ausencia",
    codificacao="onehot", escala="zscore",
)
CLASSES = ["normal", "anomaly"]
# Mesmos atributos e tratamento em todos os experimentos de SVM.
OPCOES = {
    "linear_01": ("linear", 0.1, None),
    "linear_1": ("linear", 1.0, None),
    "linear_10": ("linear", 10.0, None),
    "linear_1_balanceada": ("linear", 1.0, "balanced"),
    "rbf_1": ("rbf", 1.0, None),
    "rbf_10": ("rbf", 10.0, None),
    "rbf_10_balanceada": ("rbf", 10.0, "balanced"),
}


def avaliar_divisao(x, y, divisao, opcoes):
    fold, treino, validacao = divisao
    x_treino, x_validacao, y_treino = transformar(x, y, treino, validacao, PREPARACAO)
    y_validacao = y[validacao]
    assert np.isfinite(x_treino).all() and np.isfinite(x_validacao).all()
    assert x_treino.shape[1] == x_validacao.shape[1]
    linhas = []
    for nome in opcoes:
        kernel, c, pesos = OPCOES[nome]
        if kernel == "linear":
            modelo = LinearSVC(
                C=c, class_weight=pesos, dual=False, max_iter=5_000, random_state=SEMENTE,
            )
        else:
            modelo = SVC(
                C=c, class_weight=pesos, kernel="rbf", gamma="scale",
                cache_size=256, max_iter=200_000,
            )
        inicio = perf_counter()
        modelo.fit(x_treino, y_treino)
        tempo_ajuste = perf_counter() - inicio
        inicio = perf_counter()
        previsto = modelo.predict(x_validacao)
        tempo_previsao = perf_counter() - inicio
        convergiu = (
            modelo.n_iter_ < modelo.max_iter if kernel == "linear" else modelo.fit_status_ == 0
        )
        resultado = {
            "opcao": nome, "fold": fold, "kernel": kernel, "C": c, "pesos": pesos,
            "f1_macro": f1_score(y_validacao, previsto, average="macro"),
            "recall_anomaly": recall_score(y_validacao, previsto, pos_label="anomaly", zero_division=0),
            "precisao_anomaly": precision_score(y_validacao, previsto, pos_label="anomaly", zero_division=0),
            "segundos_ajuste": round(tempo_ajuste, 3), "segundos_previsao": round(tempo_previsao, 3),
            "colunas": x_treino.shape[1], "linhas_ajuste": len(y_treino),
            "linhas_validacao": len(y_validacao), "convergiu": bool(convergiu),
        }
        if fold == 5:
            resultado["matriz"] = confusion_matrix(y_validacao, previsto, labels=CLASSES).tolist()
            resultado["relatorio"] = classification_report(
                y_validacao, previsto, labels=CLASSES, output_dict=True, zero_division=0,
            )
        linhas.append(resultado)
        print(f"Divisão {fold}, {nome}: F1-macro {resultado['f1_macro']:.4f}", flush=True)
    return linhas


def main():
    csv, pasta = map(Path, sys.argv[1:3])
    trabalhadores = int(sys.argv[3])
    if trabalhadores not in (1, 2):
        raise ValueError("Escolha 1 ou 2 divisões em paralelo.")
    dados = carregar_treino(csv)
    y = dados["class"].to_numpy()
    representacoes, grupos = preparar_base(dados)
    x = representacoes["C"]
    divisoes = separar_divisoes(representacoes, y, grupos)

    # Paralelismo entre divisões independentes, com um thread numérico por ajuste.
    with threadpool_limits(limits=1):
        inicio = perf_counter()
        rodadas = Parallel(n_jobs=trabalhadores, prefer="threads")(
            delayed(avaliar_divisao)(x, y, divisao, OPCOES) for divisao in divisoes[:4]
        )
        tempo_comparacao = perf_counter() - inicio
        tabela = pd.DataFrame([linha for rodada in rodadas for linha in rodada])
        medias = tabela.groupby("opcao", sort=False).agg(
            f1_macro=("f1_macro", "mean"),
            desvio_f1=("f1_macro", "std"),
            recall_anomaly=("recall_anomaly", "mean"),
            precisao_anomaly=("precisao_anomaly", "mean"),
            segundos_ajuste=("segundos_ajuste", "median"),
            segundos_previsao=("segundos_previsao", "median"),
            convergiu=("convergiu", "all"),
        ).reset_index().sort_values(
            ["convergiu", "f1_macro", "recall_anomaly"], ascending=False, kind="stable",
        )
        if not medias["convergiu"].any():
            raise RuntimeError("Nenhuma opção convergiu nas quatro divisões.")
        melhor = str(medias.iloc[0]["opcao"])
        melhor_linear = medias.loc[
            medias["convergiu"] & medias["opcao"].str.startswith("linear"), "opcao"
        ].head(1).tolist()
        candidatas = list(dict.fromkeys([melhor, *melhor_linear]))
        print(f"Configurações escolhidas antes de avaliar a quinta divisão: {candidatas}", flush=True)
        reserva = avaliar_divisao(x, y, divisoes[4], candidatas)

    tabela.to_csv(pasta / "svm_por_divisao.csv", index=False)
    resumo = {
        "medias": medias.to_dict(orient="records"),
        "reserva": reserva,
        "melhor": melhor,
        "classes": CLASSES,
        "trabalhadores": trabalhadores,
        "segundos_comparacao": round(tempo_comparacao, 2),
        "ajustes_comparacao": len(tabela),
        "divisoes": [
            {"fold": fold, "treino": len(treino), "validacao": len(val),
             "anomalias": int(np.count_nonzero(y[val] == "anomaly"))}
            for fold, treino, val in divisoes
        ],
    }
    (pasta / "svm_resultados.json").write_text(json.dumps(resumo, ensure_ascii=False), encoding="utf-8")


if __name__ == "__main__":
    main()

In [ ]:
TRABALHADORES = 2
print("Comparando sete configurações de SVM em CPU.", flush=True)
subprocess.run(
    [sys.executable, "-u", str(codigo_experimentos), str(treino_csv), str(trabalho), str(TRABALHADORES)],
    check=True, env=ambiente,
)
svm = json.loads((trabalho / "svm_resultados.json").read_text(encoding="utf-8"))

## Resultados

A primeira tabela usa a média das quatro divisões e o desvio padrão amostral do F1-macro. Os tempos de ajuste e previsão são medianas dessas rodadas. A segunda tabela mostra a quinta divisão separadamente. A melhor opção da busca é escolhida antes de consultar essa divisão. A matriz de confusão e o relatório por classe detalham as previsões da opção escolhida nessa avaliação interna. O teste oficial possui entradas sem rótulos e suas métricas dependem do gabarito da professora.

In [ ]:
def tabela(colunas, linhas):
    linhas = [colunas, ["---"] * len(colunas), *linhas]
    display(Markdown("\n".join("| " + " | ".join(map(str, linha)) + " |" for linha in linhas)))

nomes = {
    "linear_01": "Linear, C=0,1", "linear_1": "Linear, C=1", "linear_10": "Linear, C=10",
    "linear_1_balanceada": "Linear, C=1, com pesos",
    "rbf_1": "RBF, C=1", "rbf_10": "RBF, C=10", "rbf_10_balanceada": "RBF, C=10, com pesos",
}
tabela(
    ("Configuração", "F1-macro médio ± desvio", "Recall de anomaly", "Precisão de anomaly", "Ajuste (s)", "Previsão (s)", "Convergiu"),
    [(nomes[r["opcao"]], f"{r['f1_macro']:.4f} ± {r['desvio_f1']:.4f}", f"{r['recall_anomaly']:.2%}",
      f"{r['precisao_anomaly']:.2%}", f"{r['segundos_ajuste']:.2f}", f"{r['segundos_previsao']:.2f}",
      "Sim" if r["convergiu"] else "Não") for r in svm["medias"]],
)
print("Melhor opção da busca:", nomes[svm["melhor"]])
tabela(
    ("Configuração", "F1-macro na quinta divisão", "Recall de anomaly", "Precisão de anomaly", "Convergiu"),
    [(nomes[r["opcao"]], f"{r['f1_macro']:.4f}", f"{r['recall_anomaly']:.2%}",
      f"{r['precisao_anomaly']:.2%}", "Sim" if r["convergiu"] else "Não") for r in svm["reserva"]],
)
print(f"{svm['ajustes_comparacao']} ajustes na busca em {svm['segundos_comparacao']:.1f} s.")
print("CSV de resultados por divisão:", trabalho / "svm_por_divisao.csv")

In [ ]:
melhor = next(r for r in svm["reserva"] if r["opcao"] == svm["melhor"])
classes = svm["classes"]
relatorio = melhor["relatorio"]
display(Markdown("### Matriz de confusão na quinta divisão"))
tabela(
    ("Classe real", *(f"Previsto {c}" for c in classes)),
    [(classe, *linha) for classe, linha in zip(classes, melhor["matriz"])],
)
tabela(
    ("Classe", "Precisão", "Recall", "F1", "Registros reais"),
    [(c, f"{relatorio[c]['precision']:.2%}", f"{relatorio[c]['recall']:.2%}",
      f"{relatorio[c]['f1-score']:.4f}", int(relatorio[c]["support"])) for c in classes],
)
print(f"F1-macro {melhor['f1_macro']:.4f} · Acurácia {relatorio['accuracy']:.2%}")

## Próxima comparação

As métricas acima são de uma avaliação interna e de uma busca pequena. A configuração escolhida é uma candidata para comparar com a árvore e a rede nas mesmas divisões. Os pesos são avaliados junto com o recall e a precisão porque detectar mais anomalias pode aumentar os falsos alarmes.

A execução em GPU seria uma comparação de implementação. A versão aqui usa as implementações de CPU do scikit-learn. Uma versão com CUDA pode usar cuML em um ambiente NVIDIA compatível e deve ser comparada nas mesmas divisões antes de qualquer conclusão sobre tempo ou qualidade.

[SVC no scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) ·
[LinearSVC no scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html) ·
[Paralelismo no scikit-learn](https://scikit-learn.org/stable/computing/parallelism.html) ·
[SVM no cuML](https://docs.nvidia.com/cuml/latest/api/)

**Apoio de IA.** Codex auxiliou na montagem e na validação deste experimento. A equipe deve revisar a implementação e declarar esse apoio no relatório.